In [1]:
import pandas as pd

df = pd.read_csv("../data/cleaned_results.csv")

df["date"] = pd.to_datetime(df["date"])

df = df.sort_values("date")

df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,result
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False,1872,draw
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False,1873,home_win
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False,1874,home_win
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False,1875,draw
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False,1876,home_win


In [ ]:
def expected_score(rating_a, rating_b):

    return 1 / (
        1 + 10 ** ((rating_b - rating_a) / 400)
    )

In [3]:
def update_elo(
    rating_a,
    rating_b,
    score_a,
    k=20
):

    expected_a = expected_score(
        rating_a,
        rating_b
    )

    return rating_a + k * (
        score_a - expected_a
    )

In [4]:
expected_score(1500, 1500)

0.5

In [5]:
update_elo(
    1500,
    1500,
    1
)

1510.0

In [6]:
elo_ratings = {}

In [7]:
all_teams = set(
    df["home_team"]
).union(
    set(df["away_team"])
)

for team in all_teams:
    elo_ratings[team] = 1500

In [9]:
home_elos = []
away_elos = []
elo_diffs = []

In [14]:
for _, match in df.iterrows():

    home_team = match["home_team"]
    away_team = match["away_team"]

    home_rating = elo_ratings[home_team]
    away_rating = elo_ratings[away_team]

    # Save PRE-MATCH ratings

    home_elos.append(home_rating)
    away_elos.append(away_rating)
    elo_diffs.append(
        home_rating - away_rating
    )

    # Determine result

    if match["home_score"] > match["away_score"]:

        home_score = 1
        away_score = 0

    elif match["home_score"] < match["away_score"]:

        home_score = 0
        away_score = 1

    else:

        home_score = 0.5
        away_score = 0.5

    # Update ratings

    new_home = update_elo(
        home_rating,
        away_rating,
        home_score
    )

    new_away = update_elo(
        away_rating,
        home_rating,
        away_score
    )

    elo_ratings[home_team] = new_home
    elo_ratings[away_team] = new_away

In [15]:
df["home_elo"] = home_elos
df["away_elo"] = away_elos
df["elo_diff"] = elo_diffs

In [16]:
df[
    [
        "home_team",
        "away_team",
        "home_elo",
        "away_elo",
        "elo_diff"
    ]
].head()

,home_team,away_team,home_elo,away_elo,elo_diff
0,Scotland,England,1500.000000,1500.000000,0.000000
1,England,Scotland,1500.000000,1500.000000,0.000000
2,Scotland,England,1490.000000,1510.000000,-20.000000
3,England,Scotland,1499.424989,1500.575011,-1.150023
4,Scotland,England,1500.541911,1499.458089,1.083822


In [17]:
df["elo_diff"].describe()

count    49281.000000
mean         8.895858
std        158.980385
min       -862.075110
25%        -87.826692
50%          8.807330
75%        107.034293
max        821.415135
Name: elo_diff, dtype: float64

In [18]:
sorted(
    elo_ratings.items(),
    key=lambda x: x[1],
    reverse=True
)[:20]

[('Spain', 1985.4176119760966),
 ('Argentina', 1985.2558961950278),
 ('France', 1953.1596777513118),
 ('Brazil', 1913.4553881975544),
 ('England', 1892.5815667223546),
 ('Portugal', 1891.1399324697454),
 ('Germany', 1881.1430857023347),
 ('Netherlands', 1879.2350292709816),
 ('Colombia', 1875.333421442335),
 ('Japan', 1861.685514728441),
 ('Morocco', 1851.648286925075),
 ('Italy', 1845.0245078830055),
 ('Croatia', 1841.5160929819106),
 ('Belgium', 1833.8534565298257),
 ('Ecuador', 1823.259225687965),
 ('Uruguay', 1817.2429015469272),
 ('Mexico', 1815.7626140899765),
 ('Denmark', 1801.1104060011712),
 ('Iran', 1798.953436703047),
 ('Switzerland', 1797.8288418505101)]

In [57]:
df1 = pd.read_csv("../data/features_v2C.csv")

In [58]:
df1.head()

,form_diff,goal_diff_diff,scored_diff,conceded_diff,result
0,-0.333333,-1.6,-1.2,0.4,away_win
1,0.600000,3.0,1.2,-1.8,home_win
2,-0.733333,-6.2,-3.6,2.6,home_win
3,0.400000,5.2,3.0,-2.2,home_win
4,-0.400000,-2.6,-2.6,0.0,away_win


In [59]:
df1["home_elo"]=df["home_elo"]
df1["away_elo"]= df["away_elo"]
df1["elo_diff"]=df["elo_diff"]

In [54]:
df1.head()

,form_diff,goal_diff_diff,scored_diff,conceded_diff,result,home_elo,away_elo,elo_diff
0,-0.333333,-1.6,-1.2,0.4,away_win,1500.000000,1500.000000,0.000000
1,0.600000,3.0,1.2,-1.8,home_win,1500.000000,1500.000000,0.000000
2,-0.733333,-6.2,-3.6,2.6,home_win,1490.000000,1510.000000,-20.000000
3,0.400000,5.2,3.0,-2.2,home_win,1499.424989,1500.575011,-1.150023
4,-0.400000,-2.6,-2.6,0.0,away_win,1500.541911,1499.458089,1.083822


In [60]:
df1.tail()

,form_diff,goal_diff_diff,scored_diff,conceded_diff,result,home_elo,away_elo,elo_diff
47963,-0.133333,0.6,0.8,0.2,home_win,1817.393929,1891.649123,-74.255194
47964,-0.200000,0.4,0.0,-0.4,home_win,1727.881865,1676.586462,51.295403
47965,-0.066667,-0.2,0.4,0.6,home_win,1742.056641,1798.628435,-56.571795
47966,0.533333,2.0,1.6,-0.4,home_win,1978.937304,1768.227934,210.709370
47967,0.666667,2.4,1.0,-1.4,home_win,1538.657771,1359.680350,178.977421


In [61]:
df1.to_csv("../data/features_v3A.csv",index=False)

In [62]:
df2 = pd.DataFrame()

In [63]:
df2["form_diff"]= df1["form_diff"]
df2["goal_diff_diff"] = df1["goal_diff_diff"]
df2["elo_diff"] = df1["elo_diff"]

In [64]:
df2["result"] = df1["result"]

In [65]:
df2.tail()

,form_diff,goal_diff_diff,elo_diff,result
47963,-0.133333,0.6,-74.255194,home_win
47964,-0.200000,0.4,51.295403,home_win
47965,-0.066667,-0.2,-56.571795,home_win
47966,0.533333,2.0,210.709370,home_win
47967,0.666667,2.4,178.977421,home_win


In [66]:
df2.to_csv("../data/features_v3B.csv",index=False)